In [13]:
# PARAMÉTEREK

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


months = 6
pattern = ['W', 'L', 'L', 'W', 'W', 'L']  # 6 hónapos minta

start_users = 10
new_users_per_month = 3
monthly_price = 8000
base_churn = 0.20  # 20%
extra_churn_per_loss_streak = 0.05  # +5% churn per extra loss
churn_reduction_per_win = 0.05 # -5% churn per win

# SEGÉDFÜGGVÉNYEK

def calc_discount(prev_outcome, prev_prev_outcome):
    """Számolja az adott hónap kedvezményét a két előző hónap alapján."""
    if prev_outcome == 'L' and prev_prev_outcome == 'L':
        return 0.0
    elif prev_outcome == 'L':
        return 0.5
    else:
        return 1.0

In [ ]:
# SZIMULÁCIÓ

df = pd.DataFrame({
    'Month': np.arange(1, months+1),
    'Pattern': pattern
})

df['New_users'] = new_users_per_month
df.loc[0, 'Active_start'] = start_users
df['Price'] = monthly_price

discounts = []
churn_rates = []
revenues = []
active_end = []

loss_streak = 0
win_streak = 0
active_users = start_users

for i in range(months):
    outcome = df.loc[i, 'Pattern']
    prev_out = df.loc[i-1, 'Pattern'] if i > 0 else None
    prev_prev_out = df.loc[i-2, 'Pattern'] if i > 1 else None

    # Kedvezmény (discount)
    discount = calc_discount(prev_out, prev_prev_out) if i > 0 else 1.0
    discounts.append(discount)

    # Churn: nő, ha vesztő széria van, csökken ha nyerő széria van
    if outcome == 'L':
        loss_streak += 1
        win_streak = 0  # Reset
    else:
        loss_streak = 0  # Reset
        win_streak += 1  # Increment

    # Churn rate számítás
    # A max(0, loss_streak - 1) biztosítja, hogy az első 'L' után még csak 20% legyen a churn.
    # A max(0, win_streak - 1) biztosítja, hogy az első 'W' után még csak 20% legyen a churn.
    churn_rate = base_churn
    if loss_streak > 1:
        churn_rate += (loss_streak - 1) * extra_churn_per_loss_streak
    elif win_streak > 0:
        # Minden 'W' 5%-kal csökkenti a churn-t (max 5%-kal a base-hez képest, ha csak 1 W van)
        # Itt egyszerűen a win_streak-kel arányosan csökkentjük:
        churn_rate -= win_streak * extra_churn_reduction_per_win
        
    # Minimum és maximum korlát (pl. 5% és 95%)
    churn_rate = np.clip(churn_rate, 0.05, 0.95) # Limitáljuk 5% és 95% közé a reálisabb eredményért
    
    churn_rates.append(churn_rate)

    # Árbevétel
    total_users = active_users + new_users_per_month
    revenue = total_users * monthly_price * discount
    revenues.append(revenue)

    # Lemorzsolódás
    churned = int(total_users * churn_rate)
    active_next = total_users - churned
    active_end.append(active_next)

    # Következő hónap kezdő állománya
    if i < months - 1:
        df.loc[i+1, 'Active_start'] = active_next

df['Discount'] = discounts
df['Churn_rate'] = churn_rates
df['Revenue'] = revenues
df['Active_end'] = active_end
df['Cumulative_revenue'] = df['Revenue'].cumsum()


In [ ]:
# EREDMÉNYEK MEGJELENÍTÉSE

display(df[['Month','Pattern','Active_start','New_users','Discount','Churn_rate','Revenue','Active_end','Cumulative_revenue']])

plt.figure(figsize=(10,5))
plt.plot(df['Month'], df['Revenue'], marker='o', label='Havi árbevétel')
plt.plot(df['Month'], df['Cumulative_revenue'], marker='s', label='Kumulált árbevétel')
plt.title("Cashflow szimuláció (6 hónap)")
plt.xlabel("Hónap")
plt.ylabel("Ft")
plt.legend()
plt.grid(True)
plt.show()

print(f"Összes árbevétel 6 hónap alatt: {df['Revenue'].sum():,.0f} Ft")
print(f"Átlagos churn: {df['Churn_rate'].mean()*100:.1f}%")